# Landings System in PyPath

## Overview
The landings system models temporal and categorical distribution of fish landings. This notebook demonstrates landing records, seasonality, and integration with Ecosim dynamics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from pypath.core.params import (
    LandingType, LandingRecord, LandingSeasonality, LandingsData
)

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Part 1: Landing Types

Define landing categories:

In [ ]:
# Show landing types
print('Landing Types:')
for ltype in LandingType:
    print(f'  {ltype.name}: {ltype.value}')

## Part 2: Creating Landing Records

Define individual landing events:

In [ ]:
# Create landing records for Cod
landing_2024_q1 = LandingRecord(
    species='Cod',
    year=2024,
    quarter=1,
    landing_type=LandingType.COMMERCIAL,
    quantity_tonnes=250.0,
    price_per_tonne=1500.0,
    port='Gdansk'
)

landing_2024_q2 = LandingRecord(
    species='Cod',
    year=2024,
    quarter=2,
    landing_type=LandingType.COMMERCIAL,
    quantity_tonnes=320.0,
    price_per_tonne=1600.0,
    port='Gdansk'
)

print(f'Landing Q1 2024: {landing_2024_q1.quantity_tonnes}t @ {landing_2024_q1.price_per_tonne}/t')
print(f'Revenue: {landing_2024_q1.quantity_tonnes * landing_2024_q1.price_per_tonne:,.0f} EUR')

## Part 3: Landing Seasonality

Define monthly distribution of landings:

In [ ]:
# Define seasonal pattern for Cod
# Higher landings in winter (spawning season)
cod_seasonality = LandingSeasonality(
    species='Cod',
    monthly_proportions=[
        0.12,  # Jan: winter peak
        0.13,  # Feb
        0.10,  # Mar
        0.07,  # Apr: spring decline
        0.05,  # May
        0.05,  # Jun
        0.06,  # Jul: summer low
        0.06,  # Aug
        0.08,  # Sep: fall increase
        0.10,  # Oct
        0.11,  # Nov
        0.11   # Dec: winter peak
    ]
)

print(f'Cod seasonal pattern:')
for month, prop in enumerate(cod_seasonality.monthly_proportions, 1):
    print(f'  Month {month:2d}: {prop:.1%}')
print(f'\nTotal: {sum(cod_seasonality.monthly_proportions):.1%}')

## Part 4: Visualizing Seasonal Pattern

In [ ]:
# Plot seasonal pattern
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(12, 6))
plt.bar(months, cod_seasonality.monthly_proportions, color='steelblue', alpha=0.7)
plt.ylabel('Monthly Proportion')
plt.title('Cod Landings Seasonality')
plt.ylim(0, 0.15)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()

## Part 5: Landings Data Container

Manage all landing records:

In [ ]:
# Create landings data system
landings = LandingsData()

# Add records
landings.add_record(landing_2024_q1)
landings.add_record(landing_2024_q2)

# Add seasonality
landings.add_seasonality(cod_seasonality)

print(f'Landings database:')
print(f'  Total records: {landings.n_records()}')
print(f'  Species: {landings.species_list()}')
print(f'  Years: {landings.year_range()}')

## Part 6: Aggregation Functions

Query and aggregate landing data:

In [ ]:
# Aggregate by species
cod_landings = landings.by_group('Cod')
print(f'Cod landings: {cod_landings:.1f} tonnes')

# Aggregate by season
q1_landings = landings.by_season(quarter=1)
q2_landings = landings.by_season(quarter=2)
print(f'\nQuarterly landings:')
print(f'  Q1: {q1_landings:.1f} tonnes')
print(f'  Q2: {q2_landings:.1f} tonnes')

# Aggregate by type
commercial = landings.by_type(LandingType.COMMERCIAL)
print(f'\nCommercial landings: {commercial:.1f} tonnes')

## Part 7: Temporal Dispatch to Monthly Steps

Convert quarterly to monthly for Ecosim integration:

In [ ]:
# Dispatch Q1 landing to monthly steps
q1_total = landing_2024_q1.quantity_tonnes
seasonal_props = cod_seasonality.monthly_proportions[:3]  # Jan, Feb, Mar
seasonal_props = np.array(seasonal_props) / sum(seasonal_props)  # normalize

monthly_dispatch = q1_total * seasonal_props

dispatch_df = pd.DataFrame({
    'Month': ['January', 'February', 'March'],
    'Landing (t)': monthly_dispatch,
    'Proportion': seasonal_props
})

print('Q1 2024 dispatch to months:')
print(dispatch_df.to_string(index=False))

## Part 8: Integration with Ecosim Dynamics

Apply landings to simulation runs:

In [ ]:
# Example: Landing trajectory over 10 years
years = np.arange(2015, 2025)
cod_catch_trajectory = np.array([200, 220, 240, 260, 280, 300, 320, 340, 360, 340])

plt.figure(figsize=(10, 6))
plt.plot(years, cod_catch_trajectory, 'o-', linewidth=2, markersize=8, color='steelblue')
plt.fill_between(years, cod_catch_trajectory * 0.9, cod_catch_trajectory * 1.1, 
                  alpha=0.2, color='steelblue', label='±10% uncertainty')
plt.xlabel('Year')
plt.ylabel('Annual Catch (tonnes)')
plt.title('Cod Landings Trajectory - Historical and Projected')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

## Part 9: Time Series Visualization

Multi-species landing time series:

In [ ]:
# Create example time series for multiple species
years_series = np.arange(2010, 2025)

data = pd.DataFrame({
    'Year': years_series,
    'Cod': np.array([150, 160, 170, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 340, 320]),
    'Herring': np.array([400, 420, 440, 460, 480, 500, 520, 540, 560, 580, 590, 595, 600, 590, 580]),
    'Sprat': np.array([100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 210, 200])
})

plt.figure(figsize=(12, 6))
for species in ['Cod', 'Herring', 'Sprat']:
    plt.plot(data['Year'], data[species], 'o-', linewidth=2, label=species)

plt.xlabel('Year')
plt.ylabel('Annual Landings (tonnes)')
plt.title('Multi-species Landing Time Series')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

## Part 10: DataFrame Export and Analysis

In [ ]:
# Export landings to DataFrame
landings_df = landings.to_dataframe()
print('Landings database:')
print(landings_df)

# Save to CSV
landings_df.to_csv('landings.csv', index=False)
print('\nSaved to landings.csv')

## Performance Tips

- LandingsData lookup by species/year is O(1)
- Monthly dispatch uses pre-computed proportions
- Cache aggregation results for repeated queries
- Use vectorized operations for temporal interpolation